# Advanced Tutorial Problems with Solutions — Python Class Body Scope

This notebook is a **second, independent problem set** on Python class body scope.

It deliberately follows a tutorial rhythm:

- introduce one idea,
- make a prediction,
- run a very small experiment,
- explain exactly what happened,
- change one detail,
- repeat,
- then solve a larger problem that combines the ideas.

The central question throughout the notebook is:

> **When Python sees a name inside or around a class, which namespace is actually searched?**

We will focus especially on:

- execution of the class body,
- names created while the class body is running,
- functions defined inside a class,
- globals that accidentally hide bugs,
- generated classes and closures,
- list/set/dict comprehensions inside a class,
- nested classes,
- decorators,
- inheritance and `cls`,
- class-body defaults,
- and techniques for debugging scope problems.


## A note about how to use this notebook

Do not immediately run every code cell.

For each section:

1. read the question,
2. predict the output,
3. explain your prediction in words,
4. then run the code,
5. compare your mental model with Python's behavior.

The exercises are intentionally small at first. Later problems combine multiple scope rules at once.


## The mental model we will test

A `class` statement executes code.

While that code is executing, Python builds a namespace for the future class.

That explains why this works:

```python
class Example:
    x = 10
    y = x + 5
```

The expression used to compute `y` executes directly in the class body, after `x` has already been created.

But a function defined in that class body is a different story.

The function object is placed in the class namespace, but the **function's own lexical scope does not use the class namespace as an enclosing function scope**.

That distinction is the source of most of the interesting problems in this notebook.


In [1]:
import inspect
import dis

def expect_error(exc_type, fn):
    try:
        fn()
    except exc_type as exc:
        print(f"Caught expected {exc_type.__name__}: {exc}")
        return exc
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}"
        ) from exc
    else:
        raise AssertionError(
            f"Expected {exc_type.__name__}, but no exception was raised"
        )

print("Helpers ready.")


Helpers ready.


# Tutorial 1 — The class body is executable code

We begin with the simplest possible observation.

The next class does not merely *declare* names.

Python actually executes the right-hand side of each assignment while creating the class.


Before running the next cell, answer:

- What is `Sequence.a`?
- What is `Sequence.b`?
- What is `Sequence.c`?
- In what order must those expressions have been evaluated?


In [2]:
class Sequence:
    a = 2
    b = a + 3
    c = b * 4

print(Sequence.a)
print(Sequence.b)
print(Sequence.c)


2
5
20


## Solution

The output is:

```text
2
5
20
```

The assignments execute from top to bottom.

By the time Python evaluates `b = a + 3`, the name `a` already exists in the class namespace.

Likewise, `b` already exists when Python evaluates `c = b * 4`.


Now let us reverse the dependency.

What happens if the class body tries to use a name **before** that name has been assigned?


In [3]:
def define_bad_order():
    class BadOrder:
        first = second + 1
        second = 10
    return BadOrder

expect_error(NameError, define_bad_order)


Caught expected NameError: name 'second' is not defined


NameError("name 'second' is not defined")

The important point is that a class body is not a magical two-pass declaration system.

Names become available as execution reaches their assignments.


# Tutorial 2 — Direct class-body lookup vs method lookup

This is the first major scope puzzle.

We will use the same spelling, `VALUE`, in two places:

- module/global scope,
- class scope.

Then we will access `VALUE` in several different ways.


In [4]:
VALUE = "global"

class LookupDemo:
    VALUE = "class"

    direct = VALUE

    def unqualified(self):
        return VALUE

    def qualified(self):
        return self.VALUE


Pause here.

Predict these three values:

```python
LookupDemo.direct
LookupDemo().unqualified()
LookupDemo().qualified()
```

The interesting question is not only *what* they return, but *why the second line does not behave like the first*.


In [5]:
print("direct      ->", LookupDemo.direct)
print("unqualified ->", LookupDemo().unqualified())
print("qualified   ->", LookupDemo().qualified())


direct      -> class
unqualified -> global
qualified   -> class


## Solution

The results are:

```text
direct      -> class
unqualified -> global
qualified   -> class
```

`direct = VALUE` executes in the class body, so it can use the class namespace.

But `unqualified()` is a function. When the function later executes, the class namespace is not searched as an enclosing lexical scope.

`qualified()` avoids that ambiguity by explicitly performing attribute lookup through `self`.


This leads to a practical rule:

> Inside instance methods, if you mean an instance/class attribute, write `self.name`.

Do not rely on an unqualified name that merely happens to have the same spelling.


# Tutorial 3 — A missing global gives a useful error

The previous example was actually dangerous because a module-level `VALUE` existed.

Let us remove the matching global and repeat the bad lookup.


In [6]:
globals().pop("ONLY_CLASS", None)

class MissingGlobal:
    ONLY_CLASS = 123

    def bad(self):
        return ONLY_CLASS


What do you expect?

A common incorrect mental model is:

> "`ONLY_CLASS` is in the class, and the function is in the class, so the function should see it."

Let us test that model.


In [7]:
expect_error(NameError, MissingGlobal().bad)


Caught expected NameError: name 'ONLY_CLASS' is not defined


NameError("name 'ONLY_CLASS' is not defined")

## Solution

The `NameError` is useful because it exposes the mistaken lookup assumption.

The method does **not** automatically search the class namespace for an unqualified `ONLY_CLASS`.

The straightforward fix is:


In [8]:
class FixedMissingGlobal:
    ONLY_CLASS = 123

    def good(self):
        return self.ONLY_CLASS

assert FixedMissingGlobal().good() == 123
print(FixedMissingGlobal().good())


123


# Tutorial 4 — The more dangerous version: a global hides the bug

Now we intentionally create a global with the same name.

This is a realistic source of bugs because the program does not crash.


In [9]:
TIMEOUT = 999

class ApiClient:
    TIMEOUT = 15

    def timeout(self):
        return TIMEOUT


What will `ApiClient().timeout()` return?

More importantly:

> Why is this bug potentially harder to discover than a `NameError`?


In [10]:
print(ApiClient().timeout())


999


## Solution

It returns `999`.

The method's unqualified `TIMEOUT` resolves to the module-level name.

Because that name exists, Python has no reason to raise an exception.

So a scope mistake has become a **silent logic bug**.


The correction is explicit:


In [11]:
class SafeApiClient:
    TIMEOUT = 15

    def timeout(self):
        return self.TIMEOUT

assert SafeApiClient().timeout() == 15
print(SafeApiClient().timeout())


15


# Tutorial 5 — Why class methods should normally use `cls`

A class method receives the class that was actually used for the call.

That matters with inheritance.


In [12]:
class BaseParser:
    FORMAT = "base"

    @classmethod
    def format_name(cls):
        return cls.FORMAT

class JsonParser(BaseParser):
    FORMAT = "json"

class CsvParser(BaseParser):
    FORMAT = "csv"


Predict:

```python
BaseParser.format_name()
JsonParser.format_name()
CsvParser.format_name()
```


In [13]:
print(BaseParser.format_name())
print(JsonParser.format_name())
print(CsvParser.format_name())


base
json
csv


## Solution

The method is polymorphic because it uses `cls.FORMAT`.

When `JsonParser.format_name()` runs, `cls` is `JsonParser`.

This is usually better than writing `BaseParser.FORMAT` inside the method.


Let us deliberately write the less flexible version.


In [14]:
class HardCodedParser:
    FORMAT = "base"

    @classmethod
    def format_name(cls):
        return HardCodedParser.FORMAT

class SpecializedParser(HardCodedParser):
    FORMAT = "special"

print(SpecializedParser.format_name())


base


Even though the method is called on `SpecializedParser`, the hard-coded class name forces the base class attribute to be used.

That may be correct in rare cases, but it defeats normal subclass customization.


# Tutorial 6 — A static method gets no automatic class context

A static method is just a function stored on the class with special descriptor behavior.

It does not automatically receive `self` or `cls`.


In [15]:
PREFIX = "GLOBAL"

class Reporter:
    PREFIX = "REPORT"

    @staticmethod
    def build(message):
        return f"{PREFIX}: {message}"

print(Reporter.build("ready"))


GLOBAL: ready


The static method uses the global `PREFIX`.

If the operation needs class-specific configuration, a class method is usually a better fit.


In [16]:
class Reporter:
    PREFIX = "REPORT"

    @classmethod
    def build(cls, message):
        return f"{cls.PREFIX}: {message}"

class ErrorReporter(Reporter):
    PREFIX = "ERROR"

print(Reporter.build("ready"))
print(ErrorReporter.build("failed"))


REPORT: ready
ERROR: failed


## Solution lesson

Use `@staticmethod` when the operation genuinely needs no instance or class state.

Use `@classmethod` when behavior should depend on the actual class used for the call.


# Tutorial 7 — A class defined inside a function changes the picture

So far, the class has lived at module scope.

Now we place the class inside a function.

Functions create lexical scopes, so a method defined while that function is active can close over the function's locals.


In [17]:
LEVEL = "module"

def make_service():
    LEVEL = "factory"

    class Service:
        LEVEL = "class"

        def unqualified(self):
            return LEVEL

        def qualified(self):
            return self.LEVEL

    return Service

GeneratedService = make_service()


Predict:

```python
GeneratedService.LEVEL
GeneratedService().unqualified()
GeneratedService().qualified()
```

There are now **three** possible `LEVEL` values.


In [18]:
print("class attribute ->", GeneratedService.LEVEL)
print("unqualified     ->", GeneratedService().unqualified())
print("qualified       ->", GeneratedService().qualified())


class attribute -> class
unqualified     -> factory
qualified       -> class


## Solution

The values are:

```text
class attribute -> class
unqualified     -> factory
qualified       -> class
```

The method's lexical enclosing function is `make_service`, so the unqualified name can resolve to that function-local variable.

The class namespace is still not the lexical parent of the method.


# Tutorial 8 — Prove that the generated method is a closure

Rather than relying only on output, we can inspect the function.


In [19]:
closure_vars = inspect.getclosurevars(GeneratedService.unqualified)
print(closure_vars)


ClosureVars(nonlocals={'LEVEL': 'factory'}, globals={}, builtins={}, unbound=set())


Look at the `nonlocals` portion.

It should contain the captured factory variable.

That gives us direct evidence that the method closes over the surrounding **function**.


In [20]:
assert closure_vars.nonlocals["LEVEL"] == "factory"
print("Captured LEVEL:", closure_vars.nonlocals["LEVEL"])


Captured LEVEL: factory


This can be useful intentionally.

A class factory can capture construction-time configuration.

But it is important to distinguish:

- **captured configuration**, and
- **current class attributes**.

They do not necessarily stay in sync.


# Tutorial 9 — Captured value vs mutable class attribute

We will build a generated class that exposes the same initial number in two ways:

- as a closure variable,
- as a class attribute.


In [21]:
def make_counter(start):
    class Counter:
        START = start

        def captured_start(self):
            return start

        @classmethod
        def current_start(cls):
            return cls.START

    return Counter

Counter10 = make_counter(10)

print(Counter10().captured_start())
print(Counter10.current_start())


10
10


At first they agree.

Now change the class attribute:


In [22]:
Counter10.START = 500

print("captured:", Counter10().captured_start())
print("current :", Counter10.current_start())


captured: 10
current : 500


## Solution

The closure still contains `10`.

The class attribute now contains `500`.

This is not a Python bug; these are two different storage mechanisms.

A design that mixes them without a reason can become confusing.


# Tutorial 10 — Default arguments are evaluated while the `def` statement executes

A function definition is itself executed while the class body is running.

That means expressions used for default arguments can use names already present in the class namespace.


In [23]:
class Job:
    DEFAULT_RETRIES = 4

    def run(self, retries=DEFAULT_RETRIES):
        return retries

print(Job().run())


4


Why does this work even though an unqualified `DEFAULT_RETRIES` inside the method body would not?


The key is **when** the name is evaluated.

`DEFAULT_RETRIES` in the parameter default is evaluated at function-definition time, while the class body is being executed.

The integer `4` then becomes part of the function's stored defaults.


In [24]:
print(Job.run.__defaults__)
assert Job.run.__defaults__ == (4,)


(4,)


Now change the class attribute.


In [25]:
Job.DEFAULT_RETRIES = 20

print("class attribute:", Job.DEFAULT_RETRIES)
print("method default :", Job().run())


class attribute: 20
method default : 4


## Solution

The method still defaults to `4`.

The default argument is not a live reference to `Job.DEFAULT_RETRIES`.

It is a value that was computed earlier.


# Tutorial 11 — A better pattern for a dynamic class-configured default

Suppose we actually want later class changes and subclass overrides to matter.

Then we should perform the attribute lookup at call time.


In [26]:
_MISSING = object()

class DynamicJob:
    DEFAULT_RETRIES = 4

    def run(self, retries=_MISSING):
        if retries is _MISSING:
            retries = self.DEFAULT_RETRIES
        return retries

print(DynamicJob().run())

DynamicJob.DEFAULT_RETRIES = 20
print(DynamicJob().run())


4
20


This is more explicit:

- `_MISSING` tells us the caller omitted the argument;
- `self.DEFAULT_RETRIES` performs attribute lookup when the method is called.


In [27]:
class PatientJob(DynamicJob):
    DEFAULT_RETRIES = 100

assert PatientJob().run() == 100
assert PatientJob().run(3) == 3

print(PatientJob().run())


100


# Tutorial 12 — Direct list expression vs list comprehension

Now we reach one of the most surprising class-body behaviors.

We will use the same `name` spelling globally and in the class.


In [28]:
name = "GLOBAL"

class Names:
    name = "CLASS"

    repeated = [name] * 3
    comprehension = [name for _ in range(3)]


Predict both lists before running the next cell.


In [29]:
print("repeated      :", Names.repeated)
print("comprehension :", Names.comprehension)


repeated      : ['CLASS', 'CLASS', 'CLASS']
comprehension : ['GLOBAL', 'GLOBAL', 'GLOBAL']


## Solution

The direct expression uses the class-body namespace:

```python
[name] * 3
```

so it sees `"CLASS"`.

The comprehension has its own function-like execution scope for its body.

The class namespace is not its lexical enclosing scope, so the unqualified `name` resolves outward and finds `"GLOBAL"`.


# Tutorial 13 — What if the matching global does not exist?

Let us remove the global and keep only the class-local variable.


In [30]:
globals().pop("factor", None)

def define_comprehension_failure():
    class Scale:
        factor = 10
        values = [factor * i for i in range(4)]
    return Scale

expect_error(NameError, define_comprehension_failure)


Caught expected NameError: name 'factor' is not defined


NameError("name 'factor' is not defined")

The important lesson is not merely "comprehensions are weird."

The deeper lesson is:

> A class namespace is not an enclosing function scope for the code executed inside a comprehension.


# Tutorial 14 — The outer iterable can still use a class-local

There is a subtle distinction.

The iterable expression for the outermost `for` is prepared from the surrounding class-body execution.

So this can work:


In [31]:
class Squares:
    count = 5
    values = [i * i for i in range(count)]

print(Squares.values)


[0, 1, 4, 9, 16]


`range(count)` can use the class-local `count`.

But an unqualified class-local used in the **comprehension body** is a different matter.

Compare:


In [32]:
globals().pop("multiplier", None)

def define_mixed_case():
    class Mixed:
        count = 4
        multiplier = 10
        values = [multiplier * i for i in range(count)]
    return Mixed

expect_error(NameError, define_mixed_case)


Caught expected NameError: name 'multiplier' is not defined


NameError("name 'multiplier' is not defined")

Here:

- `count` is used to build the iterable and is found,
- `multiplier` is used inside the comprehension body and is not found.

This is subtle enough that clear application code should usually avoid depending on it.


# Tutorial 15 — The same idea applies to set and dict comprehensions

The behavior is not special to lists.

Comprehensions generally use their own function-like scope.


In [33]:
label = "GLOBAL-LABEL"

class MoreComprehensions:
    label = "CLASS-LABEL"

    labels_set = {label for _ in range(2)}
    labels_dict = {i: label for i in range(2)}

print(MoreComprehensions.labels_set)
print(MoreComprehensions.labels_dict)


{'GLOBAL-LABEL'}
{0: 'GLOBAL-LABEL', 1: 'GLOBAL-LABEL'}


## Solution

Both comprehension bodies resolve the unqualified `label` outside the class namespace.

If this kind of code feels surprising, that is a signal to rewrite it more explicitly.


# Tutorial 16 — A nested class is not a closure over the outer class

People often expect lexical nesting syntax to imply lexical capture.

Classes do not work that way.


In [34]:
globals().pop("secret", None)

def define_nested_failure():
    class Outer:
        secret = "outer"

        class Inner:
            copied = secret

    return Outer

expect_error(NameError, define_nested_failure)


Caught expected NameError: name 'secret' is not defined


NameError("name 'secret' is not defined")

The body of `Inner` does not automatically search the namespace being built for `Outer`.

The visual nesting expresses an attribute relationship after class creation:

```python
Outer.Inner
```

but not a function-style lexical closure.


A clearer solution is to wire the dependency explicitly after the outer class exists:


In [35]:
class Outer:
    secret = "outer"

    class Inner:
        pass

Outer.Inner.copied = Outer.secret

print(Outer.Inner.copied)
assert Outer.Inner.copied == "outer"


outer


# Tutorial 17 — Decorator expressions are evaluated during class-body execution

A decorator expression is another piece of code that runs while the class body is executing.

So a decorator can be stored in a class-local name before it is used.


In [36]:
def surround(fn):
    def wrapper(*args, **kwargs):
        return f"<<{fn(*args, **kwargs)}>>"
    return wrapper

class Decorated:
    deco = surround

    @deco
    def text(self):
        return "hello"

print(Decorated().text())


<<hello>>


This succeeds because `@deco` is evaluated while Python is executing the class body.

But that fact does **not** mean the body of `text()` sees arbitrary class locals as lexical variables.

Decorator lookup and later function execution are separate stages.


# Tutorial 18 — A decorator argument can use a previously created class-local

Let us make that timing even more obvious.


In [37]:
def tag(prefix):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            return f"{prefix}{fn(*args, **kwargs)}"
        return wrapper
    return decorator

class Tagged:
    PREFIX = "[class] "

    @tag(PREFIX)
    def message(self):
        return "ready"

print(Tagged().message())


[class] ready


`PREFIX` is resolved while Python evaluates the decorator expression in the class body.

The resulting decorator captures the string value.

Again, this is definition-time behavior, not method-body lookup.


# Tutorial 19 — Class-body expressions can capture snapshots

A class attribute can be computed from another class attribute.

But once the value is computed, later rebinding of the source attribute does not automatically recompute dependent attributes.


In [38]:
class Dimensions:
    WIDTH = 10
    HEIGHT = 5
    AREA = WIDTH * HEIGHT

print(Dimensions.AREA)

Dimensions.WIDTH = 100

print("WIDTH:", Dimensions.WIDTH)
print("AREA :", Dimensions.AREA)


50
WIDTH: 100
AREA : 50


## Solution

`AREA` remains `50`.

It was calculated during class creation.

This is conceptually similar to default arguments: a value was computed at definition time.


If the value should always reflect current attributes, compute it dynamically instead:


In [39]:
class DynamicDimensions:
    WIDTH = 10
    HEIGHT = 5

    @classmethod
    def area(cls):
        return cls.WIDTH * cls.HEIGHT

print(DynamicDimensions.area())
DynamicDimensions.WIDTH = 100
print(DynamicDimensions.area())


50
500


# Tutorial 20 — Properties are still functions

A `property` can look visually like attribute access:

```python
obj.description
```

but the getter is still a function.

Therefore the same method-body scope rules apply.


In [40]:
UNIT = "GLOBAL-UNIT"

class Weight:
    UNIT = "kg"

    def __init__(self, value):
        self.value = value

    @property
    def description(self):
        return f"{self.value} {UNIT}"

print(Weight(8).description)


8 GLOBAL-UNIT


The property accidentally uses the global.

The correction is the same as for an ordinary instance method:


In [41]:
class Weight:
    UNIT = "kg"

    def __init__(self, value):
        self.value = value

    @property
    def description(self):
        return f"{self.value} {self.UNIT}"

assert Weight(8).description == "8 kg"
print(Weight(8).description)


8 kg


# Tutorial 21 — `self.attr` participates in normal attribute lookup

Using `self.attr` is not merely a way to access an instance dictionary.

If the instance has no such attribute, lookup continues through its class and base classes.


In [42]:
class Theme:
    COLOR = "blue"

    def color(self):
        return self.COLOR

class DarkTheme(Theme):
    COLOR = "black"

print(Theme().color())
print(DarkTheme().color())


blue
black


This is why `self.COLOR` often gives the behavior you want:

- instance override if one exists,
- otherwise subclass override,
- otherwise inherited class value.


In [43]:
dark = DarkTheme()
dark.COLOR = "charcoal"

print(dark.color())
print(DarkTheme.COLOR)
print(Theme.COLOR)


charcoal
black
blue


The instance assignment shadows the class value only for that object.

This is attribute lookup, not lexical name resolution, but the distinction is central to writing scope-safe methods.


# Tutorial 22 — `cls` can create subclass-specific state

Consider a class method that increments a class attribute.


In [44]:
class CounterBase:
    total = 0

    @classmethod
    def increment(cls):
        cls.total += 1
        return cls.total

class CounterChild(CounterBase):
    pass


Predict the values after calling `CounterChild.increment()` twice.

Will `CounterBase.total` also become `2`?


In [45]:
CounterChild.increment()
CounterChild.increment()

print("base :", CounterBase.total)
print("child:", CounterChild.total)


base : 0
child: 2


## Solution

The first read can find the inherited `total = 0`.

But the assignment stores the new value on `CounterChild`, because `cls` is `CounterChild`.

So the subclass develops its own `total` attribute.


# Tutorial 23 — Watch the class namespace grow

Because the class body executes sequentially, we can inspect `locals()` during that execution.


In [46]:
class NamespaceTimeline:
    first = 1
    after_first = tuple(sorted(locals().keys()))

    second = 2
    after_second = tuple(sorted(locals().keys()))

print("after first:")
print(NamespaceTimeline.after_first)

print("\nafter second:")
print(NamespaceTimeline.after_second)


after first:
('__firstlineno__', '__module__', '__qualname__', 'first')

after second:
('__firstlineno__', '__module__', '__qualname__', 'after_first', 'first', 'second')


You should see `first` in both snapshots, but `second` only in the later snapshot.

This makes the execution model concrete:

the class namespace is being populated as statements run.


# Tutorial 24 — Bytecode can reveal the kind of lookup

We do not need bytecode to use Python well, but it can make the difference very concrete.

Compare:

```python
return LIMIT
```

with:

```python
return self.LIMIT
```


In [47]:
LIMIT = "global"

class BytecodeScope:
    LIMIT = "class"

    def unqualified(self):
        return LIMIT

    def qualified(self):
        return self.LIMIT

print("unqualified:")
dis.dis(BytecodeScope.unqualified)

print("\nqualified:")
dis.dis(BytecodeScope.qualified)


unqualified:
  6           RESUME                   0

  7           LOAD_GLOBAL              0 (LIMIT)
              RETURN_VALUE

qualified:
  9           RESUME                   0

 10           LOAD_FAST                0 (self)
              LOAD_ATTR                0 (LIMIT)
              RETURN_VALUE


The exact bytecode can vary between Python versions.

But conceptually, one path performs name/global resolution, while the other loads `self` and then performs attribute lookup.

Those are fundamentally different operations.


# Tutorial 25 — A special exception: `__class__`

There is one important compiler-supported feature that can look like class closure behavior.

A method can refer to `__class__`.


In [48]:
class DefiningClass:
    def where_defined(self):
        return __class__.__name__

class DerivedDefiningClass(DefiningClass):
    pass

print(DefiningClass().where_defined())
print(DerivedDefiningClass().where_defined())


DefiningClass
DefiningClass


Both calls return the name of the class in which the method was defined.

Let us inspect the function:


In [49]:
print(DefiningClass.where_defined.__code__.co_freevars)
print(DefiningClass.where_defined.__closure__)


('__class__',)
(<cell at 0x0000027362389120: type object at 0x0000027351D5C7C0>,)


Python creates special support for a `__class__` cell when required.

Do not generalize this special mechanism into the idea that arbitrary class variables are automatically closure variables.


# Tutorial 26 — Zero-argument `super()` is related to that special machinery

Consider:


In [50]:
class Parent:
    def describe(self):
        return "parent"

class Child(Parent):
    def describe(self):
        return super().describe() + " -> child"

print(Child().describe())
print(Child.describe.__code__.co_freevars)


parent -> child
('__class__',)


The `__class__` support used by the compiler is part of what makes zero-argument `super()` possible.

Again, this is a specific language feature, not a general rule for class locals.


# Advanced Problem 1 — Four meanings of the same spelling

We now move from guided demonstrations to larger problems.

Consider the following program.

Do **not** run it yet.


In [51]:
TOKEN = "module"

def make_token_class():
    TOKEN = "factory"

    class TokenClass:
        TOKEN = "class"

        copied = TOKEN

        def method(self):
            return TOKEN

        def attribute(self):
            return self.TOKEN

        @classmethod
        def class_attribute(cls):
            return cls.TOKEN

    return TokenClass

TokenClass = make_token_class()


## Step 1 — Predict

Write down the expected values of:

```python
TokenClass.TOKEN
TokenClass.copied
TokenClass().method()
TokenClass().attribute()
TokenClass.class_attribute()
```

There are three candidate strings:

- `"module"`
- `"factory"`
- `"class"`


In [52]:
results_1 = {
    "class attr": TokenClass.TOKEN,
    "copied": TokenClass.copied,
    "method": TokenClass().method(),
    "attribute": TokenClass().attribute(),
    "class method": TokenClass.class_attribute(),
}

for key, value in results_1.items():
    print(f"{key:12} -> {value}")


class attr   -> class
copied       -> class
method       -> factory
attribute    -> class
class method -> class


## Step 2 — Explain each result

`TokenClass.TOKEN` is simply the stored class attribute.

`copied = TOKEN` executed directly in the class body, so it used the class-local `"class"`.

`method()` uses the unqualified name from inside a function. The nearest lexical function scope is `make_token_class`, so it uses `"factory"`.

`attribute()` explicitly performs attribute lookup through the instance.

`class_attribute()` explicitly performs attribute lookup through `cls`.


In [53]:
assert results_1 == {
    "class attr": "class",
    "copied": "class",
    "method": "factory",
    "attribute": "class",
    "class method": "class",
}

print("Advanced Problem 1 passed.")


Advanced Problem 1 passed.


# Advanced Problem 2 — Refactor away an accidental closure

Suppose the factory is intended only to initialize the class.

After creation, users are allowed to modify the class configuration.

The following implementation is therefore buggy:


In [54]:
def make_rate_class(rate):
    class Rate:
        RATE = rate

        def apply(self, amount):
            return amount * rate

    return Rate

Discount = make_rate_class(0.9)

print(Discount().apply(100))
Discount.RATE = 0.5
print(Discount().apply(100))


90.0
90.0


## Step 1 — Identify the bug

After changing `Discount.RATE`, the behavior of `apply()` does not change.

Why?

Because `apply()` uses the captured factory variable `rate`, not the current class attribute.


## Step 2 — Refactor

If runtime behavior should follow class state, read through `self` or `cls`.


In [55]:
def make_rate_class(rate):
    class Rate:
        RATE = rate

        def apply(self, amount):
            return amount * self.RATE

    return Rate

Discount = make_rate_class(0.9)

assert Discount().apply(100) == 90

Discount.RATE = 0.5
assert Discount().apply(100) == 50

print(Discount().apply(100))


50.0


## Solution principle

Use closures when you intentionally want a hidden captured value.

Use attributes when the value is conceptually part of the class or instance state.


# Advanced Problem 3 — Diagnose a class-body comprehension

We want a class that stores a multiplier and builds a lookup table.

This first attempt fails:


In [56]:
globals().pop("MULT", None)

def define_table_bad():
    class Table:
        MULT = 7
        TABLE = {i: i * MULT for i in range(5)}
    return Table

expect_error(NameError, define_table_bad)


Caught expected NameError: name 'MULT' is not defined


NameError("name 'MULT' is not defined")

## Step 1 — Why does it fail?

`MULT` is used inside the dict-comprehension body.

That body does not treat the class namespace as a lexical enclosing scope.


## Step 2 — Rewrite clearly

One simple solution is to create the class first and build the derived data afterward.


In [57]:
class Table:
    MULT = 7

Table.TABLE = {
    i: i * Table.MULT
    for i in range(5)
}

print(Table.TABLE)


{0: 0, 1: 7, 2: 14, 3: 21, 4: 28}


## Step 3 — Alternative design

If the table should reflect future subclass overrides, computing it once may still be the wrong abstraction.

A class method can compute it dynamically.


In [58]:
class DynamicTable:
    MULT = 7

    @classmethod
    def table(cls, n=5):
        return {
            i: i * cls.MULT
            for i in range(n)
        }

class DoubleTable(DynamicTable):
    MULT = 2

print(DynamicTable.table())
print(DoubleTable.table())


{0: 0, 1: 7, 2: 14, 3: 21, 4: 28}
{0: 0, 1: 2, 2: 4, 3: 6, 4: 8}


# Advanced Problem 4 — Nested class inside a factory

This problem separates two kinds of nesting:

1. function nesting,
2. class nesting.

Predict what `Inner.read()` returns.


In [59]:
VALUE = "module"

def outer_factory():
    VALUE = "factory"

    class Outer:
        VALUE = "outer-class"

        class Inner:
            VALUE = "inner-class"

            @classmethod
            def read(cls):
                return VALUE

    return Outer

OuterGenerated = outer_factory()

print(OuterGenerated.Inner.VALUE)
print(OuterGenerated.Inner.read())


inner-class
factory


## Solution

`OuterGenerated.Inner.VALUE` is `"inner-class"`.

But `Inner.read()` returns `"factory"`.

The `read` function can close over the enclosing **function** scope.

Neither `Inner`'s class namespace nor `Outer`'s class namespace becomes a lexical enclosing scope for the method.


In [60]:
info = inspect.getclosurevars(OuterGenerated.Inner.read)
print(info)
assert info.nonlocals["VALUE"] == "factory"


ClosureVars(nonlocals={'VALUE': 'factory'}, globals={}, builtins={}, unbound=set())


# Advanced Problem 5 — Definition-time default vs subclass configuration

Consider a base class with a default mode.


In [61]:
class Processor:
    MODE = "safe"

    def process(self, mode=MODE):
        return mode

class FastProcessor(Processor):
    MODE = "fast"

print(Processor().process())
print(FastProcessor().process())


safe
safe


## Step 1 — Why does the subclass still use `"safe"`?

The default argument was created when `Processor.process` was defined.

It is not reevaluated for each subclass.


## Step 2 — Make the default polymorphic


In [62]:
_MISSING_MODE = object()

class Processor:
    MODE = "safe"

    def process(self, mode=_MISSING_MODE):
        if mode is _MISSING_MODE:
            mode = self.MODE
        return mode

class FastProcessor(Processor):
    MODE = "fast"

assert Processor().process() == "safe"
assert FastProcessor().process() == "fast"

print(Processor().process())
print(FastProcessor().process())


safe
fast


# Advanced Problem 6 — A decorator captures a class-body snapshot

We can combine class-body lookup with a decorator factory.


In [63]:
def prefix_with(prefix):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            return prefix + fn(*args, **kwargs)
        return wrapper
    return decorator

class SnapshotDecorator:
    PREFIX = "A: "

    @prefix_with(PREFIX)
    def message(self):
        return "hello"

print(SnapshotDecorator().message())

SnapshotDecorator.PREFIX = "B: "

print(SnapshotDecorator().message())


A: hello
A: hello


## Solution

The decorator expression used `"A: "` while the class body was executing.

The resulting wrapper captured that string.

Changing the class attribute later does not rewrite the closure created by the decorator.


If current class state should matter at call time, design the wrapper or method to perform attribute lookup at call time instead of capturing a definition-time snapshot.


# Advanced Problem 7 — The worst kind of comprehension bug

A matching global can make a broken class-body comprehension appear to work.


In [64]:
SCALE = 100

class Suspicious:
    SCALE = 3
    values = [SCALE * i for i in range(4)]

print(Suspicious.values)


[0, 100, 200, 300]


A reader may reasonably expect:

```python
[0, 3, 6, 9]
```

but the comprehension body uses the global `SCALE`.


In [65]:
assert Suspicious.values == [0, 100, 200, 300]
print("Actual:", Suspicious.values)


Actual: [0, 100, 200, 300]


## Solution lesson

This is analogous to the silent method bug we saw earlier.

A same-named global can transform a scope mistake from an exception into incorrect output.

That is one reason these issues deserve tests.


# Advanced Problem 8 — Use inspection to classify dependencies

Given a method, can we determine whether an unqualified name is:

- global,
- nonlocal,
- built-in,
- or unbound?

Let us inspect two methods.


In [66]:
GLOBAL_SETTING = "global-setting"

class GlobalUser:
    def read(self):
        return GLOBAL_SETTING

def factory():
    LOCAL_SETTING = "factory-setting"

    class LocalUser:
        def read(self):
            return LOCAL_SETTING

    return LocalUser

LocalUser = factory()


In [67]:
print("GlobalUser.read:")
print(inspect.getclosurevars(GlobalUser.read))

print("\nLocalUser.read:")
print(inspect.getclosurevars(LocalUser.read))


GlobalUser.read:
ClosureVars(nonlocals={}, globals={'GLOBAL_SETTING': 'global-setting'}, builtins={}, unbound=set())

LocalUser.read:
ClosureVars(nonlocals={'LOCAL_SETTING': 'factory-setting'}, globals={}, builtins={}, unbound=set())


## Solution

`GlobalUser.read` reports `GLOBAL_SETTING` as a global dependency.

`LocalUser.read` reports `LOCAL_SETTING` as a nonlocal dependency.

This is a useful debugging technique when generated functions or classes have surprising name resolution.


# Advanced Problem 9 — Code review: find every scope smell

Read this class without running it.

How many scope/design problems can you identify?


In [68]:
MODE = "GLOBAL"
PREFIX = "GLOBAL: "
RATE = 100

class Payment:
    MODE = "CARD"
    PREFIX = "PAY: "
    RATE = 2

    preview = [RATE * i for i in range(3)]

    def mode(self):
        return MODE

    @staticmethod
    def message(text):
        return PREFIX + text

    def total(self, amount, rate=RATE):
        return amount * rate


## Step 1 — Problem inventory

There are at least four important issues.

### Issue A

`mode()` uses an unqualified name, so it reads the global `MODE`.

### Issue B

`message()` is a static method even though its behavior apparently depends on class configuration.

### Issue C

`rate=RATE` captures a definition-time value and will not follow later overrides.

### Issue D

The class-body comprehension uses the global `RATE` in its body rather than the class-local value.


## Step 2 — Refactor the design


In [69]:
_MISSING_RATE = object()

class Payment:
    MODE = "CARD"
    PREFIX = "PAY: "
    RATE = 2

    def mode(self):
        return self.MODE

    @classmethod
    def message(cls, text):
        return cls.PREFIX + text

    def total(self, amount, rate=_MISSING_RATE):
        if rate is _MISSING_RATE:
            rate = self.RATE
        return amount * rate

    @classmethod
    def preview(cls):
        return [cls.RATE * i for i in range(3)]


## Step 3 — Test subclass behavior


In [70]:
class CryptoPayment(Payment):
    MODE = "CRYPTO"
    PREFIX = "CRYPTO: "
    RATE = 5

assert CryptoPayment().mode() == "CRYPTO"
assert CryptoPayment.message("ok") == "CRYPTO: ok"
assert CryptoPayment().total(10) == 50
assert CryptoPayment.preview() == [0, 5, 10]

print("Refactored design behaves polymorphically.")


Refactored design behaves polymorphically.


# Advanced Problem 10 — Full prediction puzzle

This is the final large prediction exercise.

It combines nearly every rule from the notebook.

Read it slowly.


In [71]:
NAME = "module"

def build():
    NAME = "factory"

    class Puzzle:
        NAME = "class"

        copied = NAME
        repeated = [NAME] * 2
        comprehension = [NAME for _ in range(2)]

        def plain(self):
            return NAME

        def attribute(self):
            return self.NAME

        @classmethod
        def class_lookup(cls):
            return cls.NAME

        def defaulted(self, value=NAME):
            return value

    return Puzzle

Puzzle = build()


## Step 1 — Predict every value

Fill these in on paper first:

```python
Puzzle.NAME                 -> ?
Puzzle.copied               -> ?
Puzzle.repeated             -> ?
Puzzle.comprehension        -> ?
Puzzle().plain()            -> ?
Puzzle().attribute()        -> ?
Puzzle.class_lookup()       -> ?
Puzzle().defaulted()        -> ?
```


In [72]:
final_results = {
    "NAME": Puzzle.NAME,
    "copied": Puzzle.copied,
    "repeated": Puzzle.repeated,
    "comprehension": Puzzle.comprehension,
    "plain": Puzzle().plain(),
    "attribute": Puzzle().attribute(),
    "class_lookup": Puzzle.class_lookup(),
    "defaulted": Puzzle().defaulted(),
}

for key, value in final_results.items():
    print(f"{key:14} -> {value}")


NAME           -> class
copied         -> class
repeated       -> ['class', 'class']
comprehension  -> ['factory', 'factory']
plain          -> factory
attribute      -> class
class_lookup   -> class
defaulted      -> class


## Step 2 — Full solution

### `Puzzle.NAME`

This is the stored class attribute:

```python
"class"
```

### `Puzzle.copied`

The expression executed directly in the class body, so it used the class-local name:

```python
"class"
```

### `Puzzle.repeated`

This is also a direct class-body expression:

```python
["class", "class"]
```

### `Puzzle.comprehension`

The comprehension body has function-like scope.

Because `build()` is an enclosing function and contains `NAME = "factory"`, the comprehension can use that nonlocal value:

```python
["factory", "factory"]
```

### `Puzzle().plain()`

The method is also lexically enclosed by `build()`, not by the class namespace:

```python
"factory"
```

### `Puzzle().attribute()`

Explicit attribute lookup:

```python
"class"
```

### `Puzzle.class_lookup()`

Explicit class attribute lookup through `cls`:

```python
"class"
```

### `Puzzle().defaulted()`

The default expression `value=NAME` was evaluated while the `def` statement executed in the class body:

```python
"class"
```


In [73]:
expected_final = {
    "NAME": "class",
    "copied": "class",
    "repeated": ["class", "class"],
    "comprehension": ["factory", "factory"],
    "plain": "factory",
    "attribute": "class",
    "class_lookup": "class",
    "defaulted": "class",
}

assert final_results == expected_final
print("Final prediction puzzle passed.")


Final prediction puzzle passed.


# Final refactoring exercise — Make the behavior obvious

The prediction puzzle is excellent for learning scope rules.

It is not necessarily excellent application design.

Now rewrite the design so that a reader does not need to remember subtle scope rules.


In [74]:
def build_clear(name):
    class ClearPuzzle:
        NAME = name

        @classmethod
        def copied(cls):
            return cls.NAME

        @classmethod
        def repeated(cls, n=2):
            return [cls.NAME] * n

        @classmethod
        def generated_list(cls, n=2):
            return [cls.NAME for _ in range(n)]

        def plain(self):
            return self.NAME

        @classmethod
        def class_lookup(cls):
            return cls.NAME

        def defaulted(self, value=None):
            if value is None:
                value = self.NAME
            return value

    return ClearPuzzle

ClearPuzzle = build_clear("class")


In [75]:
assert ClearPuzzle.copied() == "class"
assert ClearPuzzle.repeated() == ["class", "class"]
assert ClearPuzzle.generated_list() == ["class", "class"]
assert ClearPuzzle().plain() == "class"
assert ClearPuzzle.class_lookup() == "class"
assert ClearPuzzle().defaulted() == "class"

print("Clear version passed.")


Clear version passed.


The refactored version makes the data source explicit.

A useful design guideline is:

> If a value is conceptually class state, access it as class state.

Closures and definition-time snapshots are powerful, but they should be intentional rather than accidental.


# Review — Classify each expression by *when* it executes

One of the easiest ways to reason about class scope is to ask:

> **When is this expression evaluated?**

Consider:

```python
class Demo:
    A = 10
    B = A + 1

    @decorator(A)
    def f(self, x=A):
        return A
```

These occurrences of `A` are not all equivalent.


## `B = A + 1`

Evaluated directly while the class body runs.

It can use the class-local `A`.


## `@decorator(A)`

The decorator expression is also evaluated while the class body runs.

It can use the class-local `A`.


## `x=A`

The default expression is evaluated when the `def` statement executes, still during class-body execution.

It can use the class-local `A`, and the resulting value becomes a stored function default.


## `return A`

This runs later, when the function is called.

Now normal function lexical lookup applies.

The class namespace is not automatically searched for the unqualified name.


That timing-based method solves many class-scope puzzles without memorizing isolated exceptions.


# Challenge Ladder — Short problems, full explanations

These final drills are shorter but increasingly subtle.


## Challenge A — Earlier class local

Predict:


In [76]:
class ChallengeA:
    x = 3
    y = x ** 2

print(ChallengeA.y)


9


### Solution A

`9`.

`y` is computed directly in the class body after `x` exists.


## Challenge B — Unqualified method name with no global

Predict the error:


In [77]:
globals().pop("scope_b", None)

class ChallengeB:
    scope_b = "class"

    def read(self):
        return scope_b

expect_error(NameError, ChallengeB().read)


Caught expected NameError: name 'scope_b' is not defined


NameError("name 'scope_b' is not defined")

### Solution B

The method does not search the class namespace for the unqualified name.


## Challenge C — Same spelling, global exists


In [78]:
scope_c = "global"

class ChallengeC:
    scope_c = "class"

    def read(self):
        return scope_c

print(ChallengeC().read())


global


### Solution C

`"global"`.

The matching global prevents a `NameError` and makes the bug silent.


## Challenge D — Closure beats global


In [79]:
scope_d = "global"

def make_d():
    scope_d = "factory"

    class ChallengeD:
        scope_d = "class"

        def read(self):
            return scope_d

    return ChallengeD

ChallengeD = make_d()

print(ChallengeD().read())


factory


### Solution D

`"factory"`.

The enclosing function local is nearer than the global in normal lexical lookup.


## Challenge E — Explicit attribute wins by design


In [80]:
scope_e = "global"

def make_e():
    scope_e = "factory"

    class ChallengeE:
        scope_e = "class"

        def read(self):
            return self.scope_e

    return ChallengeE

ChallengeE = make_e()

print(ChallengeE().read())


class


### Solution E

`"class"`.

The method does not perform lexical lookup for `scope_e` at all.

It explicitly performs attribute lookup through `self`.


## Challenge F — Definition-time class-local snapshot


In [81]:
class ChallengeF:
    setting = 8

    def read(self, value=setting):
        return value

ChallengeF.setting = 99

print(ChallengeF().read())


8


### Solution F

`8`.

The default value was computed when the function was defined.


## Challenge G — Dynamic lookup


In [82]:
_SENTINEL_G = object()

class ChallengeG:
    setting = 8

    def read(self, value=_SENTINEL_G):
        if value is _SENTINEL_G:
            value = self.setting
        return value

ChallengeG.setting = 99

print(ChallengeG().read())


99


### Solution G

`99`.

The attribute is read at call time.


## Challenge H — Comprehension inside a factory


In [83]:
scope_h = "global"

def make_h():
    scope_h = "factory"

    class ChallengeH:
        scope_h = "class"
        values = [scope_h for _ in range(2)]

    return ChallengeH

ChallengeH = make_h()
print(ChallengeH.values)


['factory', 'factory']


### Solution H

```python
["factory", "factory"]
```

The comprehension body can close over the function-local `scope_h`.

It does not use the class-local `scope_h`.


# Capstone Tutorial — Build a scope-safe configuration system

We will finish by designing something practical.

We want generated service classes with three properties:

1. a factory chooses the initial configuration,
2. subclasses can override that configuration,
3. runtime methods always use the current effective class configuration.

This means the factory should use its arguments only to initialize class attributes.

Methods should then use `self` or `cls`.


## Step 1 — Build the base class


In [84]:
class ServiceBase:
    NAME = "base"
    TIMEOUT = 1
    RETRIES = 0

    @classmethod
    def configuration(cls):
        return {
            "name": cls.NAME,
            "timeout": cls.TIMEOUT,
            "retries": cls.RETRIES,
        }

    def request_plan(self):
        return (
            f"{self.NAME}: "
            f"timeout={self.TIMEOUT}, "
            f"retries={self.RETRIES}"
        )


## Step 2 — Build a class factory

The function arguments exist in the factory scope.

We use them while the class body executes to initialize attributes.


In [85]:
def make_service(name, timeout, retries):
    class GeneratedService(ServiceBase):
        NAME = name
        TIMEOUT = timeout
        RETRIES = retries

    GeneratedService.__name__ = f"{name.title()}Service"
    return GeneratedService


## Step 3 — Generate multiple independent classes


In [86]:
SearchService = make_service("search", 3, 2)
BillingService = make_service("billing", 10, 5)

print(SearchService.configuration())
print(BillingService.configuration())


{'name': 'search', 'timeout': 3, 'retries': 2}
{'name': 'billing', 'timeout': 10, 'retries': 5}


## Step 4 — Verify runtime behavior uses attributes, not hidden closure values


In [87]:
SearchService.TIMEOUT = 8

print(SearchService().request_plan())

assert "timeout=8" in SearchService().request_plan()


search: timeout=8, retries=2


If `request_plan()` had used the captured factory variable directly, changing the class attribute would not have affected runtime behavior.

Our design intentionally separates:

- initialization through the factory,
- runtime lookup through attributes.


## Step 5 — Verify subclass overrides


In [88]:
class AggressiveSearch(SearchService):
    TIMEOUT = 1
    RETRIES = 10

print(AggressiveSearch.configuration())
print(AggressiveSearch().request_plan())

assert AggressiveSearch.configuration()["timeout"] == 1
assert AggressiveSearch.configuration()["retries"] == 10


{'name': 'search', 'timeout': 1, 'retries': 10}
search: timeout=1, retries=10


## Capstone solution principle

This design uses scope rules rather than fighting them.

The factory's lexical variables are useful during class construction.

After construction, configuration becomes ordinary class state.

Runtime methods use `self` and `cls`, so inheritance and later changes behave predictably.


# Final Reference — A compact scope checklist

When a name surprises you inside a class, ask these questions in order.

### 1. Is the expression executing directly in the class body?

If yes, it may see names already created in the class namespace.

### 2. Is the expression inside a method/function body?

If yes, use normal function lexical lookup rules. Do not assume the class namespace is an enclosing scope.

### 3. Is there an enclosing function around the class?

If yes, methods and comprehensions may close over that function's locals.

### 4. Is this a comprehension?

Remember that the comprehension body has function-like scope.

### 5. Is the expression a default argument or decorator expression?

Those are evaluated while the `def` statement/class body is executing.

### 6. Do you actually mean class or instance state?

If yes, prefer explicit attribute access:

- `self.attr`
- `cls.attr`

### 7. Could a same-named global be hiding a bug?

Temporarily remove or rename the global when debugging.

### 8. Is behavior expected to follow subclass overrides?

Prefer `self.attr` or `cls.attr` rather than hard-coded class names or captured definition-time values.


# Final self-test

You have mastered this topic when you can explain, without running code:

- why `B = A + 1` can see a class-local `A`,
- why `return A` inside a method usually cannot,
- why a matching global can make the error silent,
- why a method in a class factory can capture the factory's local variables,
- why a comprehension in a class behaves differently from a simple list expression,
- why a method default can freeze a class-body value,
- why `cls.attr` supports subclass polymorphism,
- why a nested class is not a lexical closure over its outer class,
- and why `__class__` is a special compiler-supported case rather than the general rule.


In [89]:
print("Tutorial notebook completed successfully.")


Tutorial notebook completed successfully.
